In [1]:
"""
Hull Tactical Market Prediction — Submit-ready baseline

What this does
--------------
- Loads /kaggle/input/hull-tactical-market-prediction/train.csv
- Adds lagged labels (lagged_forward_returns, lagged_risk_free_rate, lagged_market_forward_excess_returns)
- Feature filter: keep columns with >=20% non-missing and non-zero variance (drop 'date_id', targets, 'is_scored')
- Pipeline: MedianImputer -> StandardScaler -> ElasticNet(alpha=5e-4, l1_ratio=0.2)
- Walk-forward CV (≈180 trading days per fold, with a small gap) to pick mapping slope k for position = clip(1 + k*y_pred, 0, 2)
- Vol cap: scale (position-1) so realized vol vs forward_returns <= 1.2× market vol during CV
- Starts the official evaluation server and implements predict(data_batch)

Notes
-----
- The server must be started within ~15 minutes; training is done lazily at the first `predict` call (as permitted).
- Runtime is small for ~9k rows (safe vs the time limits).
"""

"\nHull Tactical Market Prediction — Submit-ready baseline\n\nWhat this does\n--------------\n- Loads /kaggle/input/hull-tactical-market-prediction/train.csv\n- Adds lagged labels (lagged_forward_returns, lagged_risk_free_rate, lagged_market_forward_excess_returns)\n- Feature filter: keep columns with >=20% non-missing and non-zero variance (drop 'date_id', targets, 'is_scored')\n- Pipeline: MedianImputer -> StandardScaler -> ElasticNet(alpha=5e-4, l1_ratio=0.2)\n- Walk-forward CV (≈180 trading days per fold, with a small gap) to pick mapping slope k for position = clip(1 + k*y_pred, 0, 2)\n- Vol cap: scale (position-1) so realized vol vs forward_returns <= 1.2× market vol during CV\n- Starts the official evaluation server and implements predict(data_batch)\n\nNotes\n-----\n- The server must be started within ~15 minutes; training is done lazily at the first `predict` call (as permitted).\n- Runtime is small for ~9k rows (safe vs the time limits).\n"

In [2]:

from __future__ import annotations
import os
import numpy as np
import pandas as pd
from typing import List, Tuple, Optional

# Polars is imported by the gateway; we handle it gracefully for input batches.
try:
    import polars as pl  # noqa: F401
except Exception:
    pl = None  # not required for modeling

import warnings
warnings.filterwarnings("ignore")

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet

# Kaggle evaluation API
import kaggle_evaluation.default_inference_server

# ---------------------------
# Config
# ---------------------------

DATASET_DIR = "/kaggle/input/hull-tactical-market-prediction/"
TRAIN_CSV = os.path.join(DATASET_DIR, "train.csv")

# Walk-forward & mapping settings (aligned with your EDA)
VAL_SIZE = 180     # ~ 6 months
N_FOLDS = 5
GAP = 5            # small gap to reduce leakage
CAP_RATIO = 1.2    # cap portfolio vol at ~1.2x market vol
RANDOM_STATE = 42


# ---------------------------
# Utility functions
# ---------------------------

def build_lagged_labels(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values("date_id").copy()
    for col in ["forward_returns", "risk_free_rate", "market_forward_excess_returns"]:
        df[f"lagged_{col}"] = df[col].shift(1)
    return df

def pick_feature_columns(df: pd.DataFrame, min_non_missing_ratio: float = 0.2) -> List[str]:
    """
    Keep columns with enough coverage and non-zero variance.
    Exclude targets, 'date_id', and 'is_scored' (test-only).
    """
    exclude = {"market_forward_excess_returns", "forward_returns", "risk_free_rate", "date_id", "is_scored"}
    cols = [c for c in df.columns if c not in exclude]
    # coverage
    non_missing = 1.0 - df[cols].isna().mean()
    keep = non_missing[non_missing >= min_non_missing_ratio].index.tolist()
    # non-zero variance
    nunique = df[keep].nunique(dropna=False)
    keep = [c for c in keep if nunique[c] > 1]
    return keep

def get_walk_forward_splits(date_ids: np.ndarray,
                            n_folds: int = N_FOLDS,
                            val_size: int = VAL_SIZE,
                            gap: int = GAP) -> List[Tuple[np.ndarray, np.ndarray]]:
    """
    Returns a list of (train_idx, val_idx) with advancing 180-day validation windows,
    ending at the most recent window. Each training split uses all data strictly before
    the validation window (minus a small gap).
    """
    N = len(date_ids)
    splits = []
    for i in range(n_folds, 0, -1):
        val_end = N - (i - 1) * val_size
        val_start = val_end - val_size
        if val_start <= 0:
            continue
        train_end = max(0, val_start - gap)
        tr = np.arange(0, train_end)
        va = np.arange(val_start, val_end)
        if len(tr) > 100 and len(va) == val_size:
            splits.append((tr, va))
    # ensure at most n_folds
    return splits[-n_folds:]

def clip_positions(pos: np.ndarray) -> np.ndarray:
    return np.clip(pos, 0.0, 2.0)

def apply_vol_cap(positions_minus_one: np.ndarray, fwd_returns: np.ndarray, cap_ratio: float) -> np.ndarray:
    """
    Scale (pos-1) down so realized vol vs forward_returns <= cap_ratio * market_vol
    (computed on the same slice).
    """
    market_vol = np.nanstd(fwd_returns)
    if market_vol <= 0 or np.isnan(market_vol):
        return positions_minus_one
    port_vol = np.nanstd(positions_minus_one * fwd_returns)
    if port_vol == 0 or np.isnan(port_vol):
        return positions_minus_one
    max_port_vol = cap_ratio * market_vol
    if port_vol <= max_port_vol:
        return positions_minus_one
    scale = max_port_vol / port_vol
    return positions_minus_one * scale


# ---------------------------
# Model wrapper
# ---------------------------

class SubmissionModel:
    """
    Encapsulates: feature selection, pipeline fit, and prediction→position mapping.
    """
    def __init__(self):
        self.feature_cols: Optional[List[str]] = None
        self.pipeline: Optional[Pipeline] = None
        self.k_: float = 1.0
        self._is_fitted: bool = False

    def fit(self, train_csv_path: str = TRAIN_CSV):
        # Load & align schema to test by adding lagged labels
        df = pd.read_csv(train_csv_path)
        df = build_lagged_labels(df)

        # Feature set
        self.feature_cols = pick_feature_columns(df, min_non_missing_ratio=0.2)

        X_all = df[self.feature_cols].copy()
        y_alpha = df["market_forward_excess_returns"].astype(np.float64).values
        r_fwd  = df["forward_returns"].astype(np.float64).values  # for vol cap during CV

        # Pipeline
        self.pipeline = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler(with_mean=True, with_std=True)),
            ("model", ElasticNet(alpha=5e-4, l1_ratio=0.2, max_iter=20000, random_state=RANDOM_STATE)),
        ])

        # Walk-forward CV splits
        date_ids = df["date_id"].values
        splits = get_walk_forward_splits(date_ids, n_folds=N_FOLDS, val_size=VAL_SIZE, gap=GAP)
        if not splits:
            # fallback if dataset is smaller
            tr = np.arange(0, max(0, len(df) - VAL_SIZE - GAP))
            va = np.arange(max(0, len(df) - VAL_SIZE), len(df))
            splits = [(tr, va)]

        # Candidate slopes for mapping y_pred → position
        k_candidates = np.concatenate([
            np.linspace(0.0, 1.5, 16),
            np.linspace(2.0, 20.0, 19),
            np.linspace(25.0, 100.0, 16),
        ])
        k_candidates = np.unique(np.round(k_candidates, 6))

        best_k_per_fold = []
        for tr_idx, va_idx in splits:
            X_tr, y_tr = X_all.iloc[tr_idx], y_alpha[tr_idx]
            X_va, y_va = X_all.iloc[va_idx], y_alpha[va_idx]
            r_va       = r_fwd[va_idx]

            self.pipeline.fit(X_tr, y_tr)
            yhat_va = self.pipeline.predict(X_va)

            # Select k maximizing avg (position-1) * market_forward_excess_returns under vol cap
            best_score, best_k = -1e18, 0.0
            for k in k_candidates:
                pos_minus_one = k * yhat_va
                pos_minus_one_capped = apply_vol_cap(pos_minus_one, r_va, cap_ratio=CAP_RATIO)
                pos = clip_positions(1.0 + pos_minus_one_capped)
                score = np.nanmean((pos - 1.0) * y_va)
                if score > best_score:
                    best_score, best_k = score, float(k)

            best_k_per_fold.append(best_k)

        # Robust aggregate (median)
        self.k_ = float(np.median(best_k_per_fold)) if best_k_per_fold else 1.0

        # Final fit on all available data (drop NaN in y if present)
        valid = ~np.isnan(y_alpha)
        self.pipeline.fit(X_all[valid], y_alpha[valid])
        self._is_fitted = True

        # Optional: print a tiny summary to the cell logs
        try:
            print(f"[fit] features={len(self.feature_cols)}  k={self.k_:.3f}  folds={len(splits)}")
        except Exception:
            pass

    def predict_positions(self, batch_df: "pd.DataFrame|pl.DataFrame") -> np.ndarray:
        assert self._is_fitted, "Model not trained yet."

        # Accept polars or pandas
        if pl is not None and isinstance(batch_df, pl.DataFrame):
            batch_df = batch_df.to_pandas()

        # Align columns; add missing expected features as NaN (imputed in pipeline)
        X = pd.DataFrame(index=batch_df.index)
        for col in self.feature_cols:
            X[col] = batch_df[col] if col in batch_df.columns else np.nan

        # Predict alpha and map to positions
        yhat = self.pipeline.predict(X)
        pos = clip_positions(1.0 + self.k_ * yhat).astype(np.float64)
        return pos


# ---------------------------
# Global model instance + predict() endpoint for the API
# ---------------------------

_MODEL: Optional[SubmissionModel] = None

def _ensure_model_loaded():
    global _MODEL
    if _MODEL is None:
        _MODEL = SubmissionModel()
        # Training is fast; we do it lazily here to ensure the server starts promptly.
        _MODEL.fit(TRAIN_CSV)

def predict(data_batch):
    """
    Evaluation API endpoint.

    Parameters
    ----------
    data_batch : pandas.DataFrame or polars.DataFrame
        Features for one or more timesteps, supplied by the gateway.

    Returns
    -------
    float | list[float] | pandas.Series
        Position(s) in [0, 2]. If a single row is provided, a scalar is acceptable;
        otherwise return a vector/Series matching the batch length.

    API timing contract (from demo):
    - First predict call can take longer (e.g., load/train).
    - Subsequent batches must be returned quickly (e.g., < ~1 minute).
    """
    _ensure_model_loaded()

    # Fast path: ensure type is pandas
    if pl is not None and isinstance(data_batch, pl.DataFrame):
        data_batch = data_batch.to_pandas()

    positions = _MODEL.predict_positions(data_batch)

    # Return scalar if single-row batch to be extra compatible
    if getattr(data_batch, "shape", None) and data_batch.shape[0] == 1:
        return float(positions[0])
    return positions


In [3]:

# ---------------------------
# Start the server (required)
# ---------------------------

inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    # In the hidden rerun, serve() must be called; the gateway container streams timesteps.
    inference_server.serve()
else:
    # For local sanity-checks in the Kaggle Notebook UI, run the gateway against the public files.
    # This will consume test.csv under DATASET_DIR and create a submission file locally.
    inference_server.run_local_gateway((DATASET_DIR,))


[fit] features=97  k=100.000  folds=5
